# MorganTrace — Análisis Exploratorio de Datos (EDA)

**Proyecto:** Detección de fraude financiero electrónico en tiempo real  
**Dataset:** IEEE-CIS Fraud Detection (Kaggle) — ~590K transacciones, 433 variables  
**Autor:** Jean Pierre Azabache  

## Objetivos
1. Analizar el desbalance de clases (fraude ~3.5% vs legítimo ~96.5%)
2. Explorar distribución de montos por tipo de transacción
3. Identificar correlaciones entre variables numéricas
4. Detectar valores nulos que requieran tratamiento


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid', palette='Set2')
print('✅ Librerías cargadas correctamente')

In [ ]:
# ── Carga del dataset ──────────────────────────────────────────────────────────
# Descargar con: kaggle competitions download -c ieee-fraud-detection -p data/raw/
try:
    df_trans = pd.read_csv('data/raw/train_transaction.csv')
    df_ident = pd.read_csv('data/raw/train_identity.csv')
    df = df_trans.merge(df_ident, on='TransactionID', how='left')
    print(f'✅ Dataset cargado: {df.shape[0]:,} transacciones, {df.shape[1]} variables')
except FileNotFoundError:
    print('⚠️  Archivos no encontrados. Ejecuta:')
    print('   kaggle competitions download -c ieee-fraud-detection -p data/raw/')
    print('   cd data/raw && unzip ieee-fraud-detection.zip')
    raise

In [ ]:
# ── Análisis del desbalance de clases ──────────────────────────────────────────
conteo = df['isFraud'].value_counts()
pct_fraude = (conteo[1] / len(df)) * 100

print(f'Transacciones legítimas  : {conteo[0]:>8,}  ({100-pct_fraude:.2f}%)')
print(f'Transacciones fraudulentas: {conteo[1]:>8,}  ({pct_fraude:.2f}%)')
print(f'Ratio de desbalance       : {conteo[0]/conteo[1]:.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores = ['#2ecc71', '#e74c3c']
axes[0].bar(['Legítima', 'Fraude'], conteo.values, color=colores, edgecolor='black')
axes[0].set_title('Distribución de Clases', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de transacciones')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(conteo.values, labels=['Legítima', 'Fraude'],
            colors=colores, autopct='%1.2f%%', startangle=90, shadow=True)
axes[1].set_title('Proporción de Fraude', fontsize=14, fontweight='bold')

plt.tight_layout()
import os; os.makedirs('notebooks/figuras', exist_ok=True)
plt.savefig('notebooks/figuras/01_desbalance_clases.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Distribución de montos: fraude vs. legítimo ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for clase, color, label in [(0, '#2ecc71', 'Legítima'), (1, '#e74c3c', 'Fraude')]:
    datos = df[df['isFraud'] == clase]['TransactionAmt']
    axes[0].hist(np.log1p(datos), bins=50, alpha=0.6, color=color, label=label)
axes[0].set_title('Distribución de Montos (escala log)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('log(Monto + 1)')
axes[0].legend()

datos_box = [
    df[df['isFraud'] == 0]['TransactionAmt'].clip(0, 2000),
    df[df['isFraud'] == 1]['TransactionAmt'].clip(0, 2000)
]
axes[1].boxplot(datos_box, labels=['Legítima', 'Fraude'],
                patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Comparación de Montos por Clase', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Monto (USD)')

plt.tight_layout()
plt.savefig('notebooks/figuras/01_distribucion_montos.png', dpi=150, bbox_inches='tight')
plt.show()

print('─' * 50)
for clase, label in [(0, 'Legítima'), (1, 'Fraude')]:
    m = df[df['isFraud'] == clase]['TransactionAmt']
    print(f'{label}: media=${m.mean():.2f} | mediana=${m.median():.2f} | max=${m.max():.2f}')

In [ ]:
# ── Mapa de calor de correlaciones (top 20 variables) ─────────────────────────
cols_num = df.select_dtypes(include=[np.number]).columns.tolist()
correlaciones = df[cols_num].corr()['isFraud'].abs().sort_values(ascending=False)
top_vars = correlaciones.head(21).index.tolist()

print('Top 20 variables más correlacionadas con isFraud:')
print(correlaciones.head(21).to_string())

fig, ax = plt.subplots(figsize=(16, 12))
matriz_corr = df[top_vars].corr()
mask = np.triu(np.ones_like(matriz_corr, dtype=bool))
sns.heatmap(matriz_corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Mapa de Calor de Correlaciones — Top 20 Variables', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/figuras/01_mapa_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Análisis de valores nulos ──────────────────────────────────────────────────
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumen = pd.DataFrame({'nulos': nulos, 'porcentaje': pct_nulos})
resumen = resumen[resumen['nulos'] > 0].sort_values('porcentaje', ascending=False)

print(f'Columnas con valores nulos: {len(resumen)} de {len(df.columns)}')
print(f'\nTop 20 columnas con más nulos:')
print(resumen.head(20).to_string())

cols_alto_nulo = resumen[resumen['porcentaje'] > 50].index.tolist()
print(f'\nColumnas con >50% nulos: {len(cols_alto_nulo)}')
print('\n✅ EDA completado. Continuar con 02_features.ipynb')